# HW1 setup (Google Colab)

Colab equivalent of `setup.sh` for this repository. Skips the
`conda` environment management in `setup.sh` — a Colab runtime is already a
single disposable environment, so there is nothing to isolate.

Use a GPU runtime: **Runtime -> Change runtime type -> T4 GPU** (or better).

## 0. Mount Google Drive

Assumes this repo already lives somewhere in your Drive (e.g. unzipped and
uploaded there beforehand). Adjust `REPO_DIR` to wherever you put it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:

REPO_DIR = "/content/drive/MyDrive/<path-to-hw1>"  # edit to match where you put the repo
%cd {REPO_DIR}

## 1. Dependencies

In [ ]:
!pip install -r requirements.txt
!pip install -e .

## 2. Data

Same Hub download `setup.sh` does — resumable, content-hashed, versioned.

In [ ]:
import os
import shutil, sys
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO = os.environ.get("HF_DATA_REPO", "aashiqmuhamed/anlp-f2026-hw1")
REV = os.environ.get("HF_REVISION", "main")
FILES = ["stories42M.pt", "tinystories-tokens.npz"]

Path("data").mkdir(exist_ok=True)
print(f"Fetching data files from {REPO}...")

for name in FILES:
    dest = Path("data") / name
    if dest.exists():
        print(f"  {dest} already present")
        continue
    try:
        cached = hf_hub_download(repo_id=REPO, filename=name,
                                  repo_type="dataset", revision=REV)
    except Exception as exc:
        print(f"  FAILED to fetch {name}: {exc}", file=sys.stderr)
        print(f"  If the download keeps failing, ask on Piazza.", file=sys.stderr)
        raise
    shutil.copy(cached, dest)
    print(f"  {dest} ({dest.stat().st_size/1e6:.0f}MB)")

## 3. Weights & Biases (needed for Part 3)

In [ ]:
!wandb login

## 4. Verify

In [ ]:
!pytest tests/ -q